# Air hockey detector and tracker on Colab

Trains the RF-DETR puck/mallet detector on the reviewed Roboflow labels, then runs it on play clips with the
tracker and renders the results: the camera view with detections and tracks, the top-down table, and plots.
Code comes from [github.com/abezh10/rlairhockey](https://github.com/abezh10/rlairhockey); the same scripts run on the cluster.

**Before you start**
1. *Runtime → Change runtime type →* a GPU. A100 or L4 is best; T4 works but trains several times slower.
2. *Secrets* (key icon, left sidebar): add `ROBOFLOW_API_KEY` (the private key) and `ROBOFLOW_WORKSPACE`
   (`abrahams-workspace-zlrim`), with notebook access on.
3. For the tracking part: the clips in Google Drive under `My Drive/rlairhockey/` (`data/clips/<video>/*.mp4` and
   `data/calib/frames/*.jpg`). From the cluster: `scripts/sync_drive.sh push`.

Training needs only steps 1-2; the labels and frames come straight from Roboflow.

In [ ]:
# Settings
NAME = "colab-v2"        # model name: weights go to models/detector/<NAME>, here and in Drive
DATASET = "v2"           # labels are pulled into data/labels/detector/<DATASET>
EPOCHS = 50
TRAIN = True             # False: skip training and use Drive's models/detector/<NAME> (e.g. "v2" from the cluster)
CLIPS = "demo"           # "demo" (one clip per match), "all", or a list of clip names
RENDER_SECONDS = 40      # render at most this much of each clip (inline videos get large)
DRIVE = "/content/drive/MyDrive/rlairhockey"

## Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os
if not os.path.exists("/content/rlairhockey"):
    !git clone -q https://github.com/abezh10/rlairhockey.git /content/rlairhockey
%cd /content/rlairhockey
!git pull -q
!pip install -q "rfdetr[train]==1.11.2" roboflow python-dotenv
!pip install -q -e . --no-deps

In [ ]:
from google.colab import drive, userdata
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
os.environ["ROBOFLOW_WORKSPACE"] = userdata.get("ROBOFLOW_WORKSPACE")
drive.mount("/content/drive")
os.makedirs(DRIVE, exist_ok=True)

## 1. Training data
Pulls every reviewed label from Roboflow (puck and mallet; images still tagged `prelabel` are left out) and downloads
the frames. Valid/test hold out about 20% of each camera framing.

In [ ]:
!python scripts/roboflow_pull.py --name {DATASET}

## 2. Train
RF-DETR Medium at 1280 px. Batch size follows GPU memory; the effective batch stays 16. About 30-40 min on an A100.

In [ ]:
import shutil, torch
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
batch, accum = (4, 4) if mem > 38 else (2, 8) if mem > 20 else (1, 16)
print(f"{torch.cuda.get_device_name(0)}, {mem:.0f} GB: batch {batch} x {accum} accumulation steps")
local, saved = f"models/detector/{NAME}", f"{DRIVE}/models/detector/{NAME}"
if TRAIN:
    !python -W ignore scripts/train_detector.py --data data/labels/detector/{DATASET} --name {NAME} --epochs {EPOCHS} --batch {batch} --accum {accum}
    os.makedirs(saved, exist_ok=True)
    for f in ("checkpoint_best_total.pth", "training_config.json"):
        shutil.copy(f"{local}/{f}", saved)
    print("weights saved to", saved)
else:
    os.makedirs(local, exist_ok=True)
    shutil.copy(f"{saved}/checkpoint_best_total.pth", local)

## 3. Evaluate on the held-out frames
Precision, recall and box-center error per class at several score thresholds (IoU >= 0.5 counts as a hit).
Most held-out labels began as v1's pre-labels, which flatters v1-like models, so the second run scores only the
benchmark frames: labeled from scratch and held out by v1 and v2 (v1 there: mallet 32/32, puck 12/14 at 0.3).

In [ ]:
!python -W ignore scripts/eval_detector.py --model {NAME} --data data/labels/detector/{DATASET}
!python -W ignore scripts/eval_detector.py --model {NAME} --data data/labels/detector/{DATASET} --frames data/labels/detector/benchmark_frames.txt

## 4. Detect and track on clips
Links the clips and calibration frames from Drive, runs the detector, then the tracker: detections are projected
onto the table, anything off the playing surface or parked (a spare puck or mallet sitting still while another is in
play) is dropped, and one puck and one mallet per half are followed.

In [ ]:
import glob, json
import pandas as pd
for v in sorted(os.listdir(f"{DRIVE}/data/clips")):
    if os.path.isdir(f"{DRIVE}/data/clips/{v}") and not os.path.exists(f"data/clips/{v}"):
        os.symlink(f"{DRIVE}/data/clips/{v}", f"data/clips/{v}")
os.makedirs("data/calib", exist_ok=True)
if not os.path.exists("data/calib/frames"):
    os.symlink(f"{DRIVE}/data/calib/frames", "data/calib/frames")

c = pd.read_csv("data/clips/clips.csv", keep_default_na=False)
c["group"] = c.video + "_" + c.angle.replace("", "nocam")
fits = json.load(open("data/calib/fits.json"))
c = c[c.group.isin(fits) & c.path.map(os.path.exists)]          # calibrated framings, clips present in Drive
if CLIPS == "demo":     # per match: the longest clip up to 40 s, else the shortest one over 10 s
    long_enough = c[c.dur >= 10].sort_values("dur")
    sel = [(g[g.dur <= 40].tail(1) if (g.dur <= 40).any() else g.head(1))["clip"].iloc[0]
           for _, g in long_enough.groupby("video")]
elif CLIPS == "all":
    sel = c["clip"].tolist()
else:
    sel = list(CLIPS)
print(len(sel), "clips:", sel[:12], "..." if len(sel) > 12 else "")
names = " ".join(sel)

In [ ]:
!python -W ignore scripts/detect_clips.py --model {NAME} --clips {names}
!python scripts/find_repeats.py --clips {names} --jobs 2
!python scripts/track_clips.py --model {NAME} --clips {names}
stats = pd.read_csv(f"data/tracks/{NAME}/stats.csv")
stats[stats["clip"].isin(sel)]

## 5. Renders
**Video:** left, the broadcast frame with every detection (colored boxes were used by the tracker, gray were rejected)
and rings where the tracked objects are; right, the top-down table with one-second trails and the puck's speed.
Yellow = puck, blue = left mallet, red = right mallet.

In [ ]:
from base64 import b64encode
from IPython.display import HTML, Image, display
show = sel[:6]
!python scripts/render_video.py {" ".join(show)} --model {NAME} --seconds {RENDER_SECONDS}
for clip in show:
    data = b64encode(open(f"outputs/videos/{NAME}/{clip}.mp4", "rb").read()).decode()
    display(HTML(f"<p><b>{clip}</b></p><video width=960 controls loop>"
                 f"<source src='data:video/mp4;base64,{data}' type='video/mp4'></video>"))

**Whole-clip paths** on the top-down table: lines are smoothed tracks, dots are detections the tracker used, gray crosses are ones it rejected.

In [ ]:
!python scripts/render_tracks.py {" ".join(show)} --model {NAME}
for clip in show:
    display(Image(f"outputs/tracks/{NAME}/{clip}.png", width=900))

**Tracking over time:** position along and across the table for each object, and the puck's speed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
meta = pd.read_csv("data/clips/clips.csv").set_index("clip")
colors = {"puck": "goldenrod", "mallet_l": "tab:blue", "mallet_r": "tab:red"}
for clip in show:
    tr, fps = pd.read_parquet(f"data/tracks/{NAME}/{clip}.parquet"), meta.loc[clip, "fps"]
    fig, ax = plt.subplots(1, 3, figsize=(17, 3.4))
    for obj, t in tr.groupby("obj"):
        ax[0].plot(t.frame / fps, t.x, color=colors[obj], label=obj)
        ax[1].plot(t.frame / fps, t.y, color=colors[obj])
    p = tr[tr.obj == "puck"]
    ax[2].plot(p.frame / fps, np.hypot(p.vx, p.vy), color=colors["puck"])
    ax[0].set(title=f"{clip}: x along the table (m)", xlabel="s"); ax[0].legend(fontsize=8)
    ax[1].set(title="y across the table (m)", xlabel="s")
    ax[2].set(title="puck speed (m/s)", xlabel="s")
    plt.tight_layout(); plt.show()

## 6. Keep the results
Copies detections, tracks and renders to Drive. To use the model on the cluster: `scripts/sync_drive.sh pull <NAME>`.

In [ ]:
for d in (f"data/detections/{NAME}", f"data/tracks/{NAME}", f"outputs/videos/{NAME}", f"outputs/tracks/{NAME}"):
    if os.path.exists(d):
        shutil.copytree(d, f"{DRIVE}/{d}", dirs_exist_ok=True)
print("saved under", DRIVE)